# OE1 · Estratificación del dataset RG6 (70 % train, 20 % valid, 10 % test)

Se toma un dataset exportado de Roboflow en formato COCO y se genera una partición estratificada lista para entrenar.

## 1) Conectar Google Drive

In [12]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2) Configuración
Se definen el dataset de entrada, el test fijo opcional y la semilla.

In [13]:
import os, re, json, shutil, hashlib, datetime
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

PROYECTO = "/content/drive/MyDrive/tesis-rg6final"

ENTRADA   = "data_roboflow"
TEST_FIJO = "data_estratificado1"

CLASES = ["rg6Bien", "rg6Mal"]
TRAIN, VALID, TEST = 0.70, 0.20, 0.10
SEMILLA = 42
UMBRAL_CASI_IGUAL = 5

INFORME = os.path.join(PROYECTO, "informe_para_codex", "oe1")
RUTA_ENTRADA = os.path.join(PROYECTO, ENTRADA)

for parte in ["train", "valid", "test"]:
    assert os.path.exists(os.path.join(RUTA_ENTRADA, parte, "_annotations.coco.json")), (
        f"Falta {parte}/_annotations.coco.json en {RUTA_ENTRADA}. "
        "¿Drive terminó de subir la carpeta? ¿Se exportó en formato COCO Segmentation?")
if TEST_FIJO:
    assert os.path.exists(os.path.join(PROYECTO, TEST_FIJO, "test", "_annotations.coco.json")), \
        f"No existe el test fijo: {TEST_FIJO}/test"
print("Entrada  :", RUTA_ENTRADA)
print("Test fijo:", TEST_FIJO or "no (reparto normal 70/20/10)")

Entrada  : /content/drive/MyDrive/tesis-rg6final/data_roboflow
Test fijo: data_estratificado1


## 3) Huella y carpeta de salida
Se calcula una huella SHA-256 para identificar la partición y no repetirla.

In [14]:
def sha256_archivo(ruta):
    with open(ruta, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

piezas = [sha256_archivo(os.path.join(RUTA_ENTRADA, p, "_annotations.coco.json")) for p in ["train", "valid", "test"]]
if TEST_FIJO:
    piezas.append(sha256_archivo(os.path.join(PROYECTO, TEST_FIJO, "test", "_annotations.coco.json")))
piezas.append(json.dumps([CLASES, TRAIN, VALID, TEST, SEMILLA]))
HUELLA = hashlib.sha256("|".join(piezas).encode()).hexdigest()

numeros = sorted(int(m.group(1)) for d in os.listdir(PROYECTO)
                 if (m := re.fullmatch(r"data_estratificado(\d+)", d)))
for n in numeros:
    ruta_origen = os.path.join(PROYECTO, f"data_estratificado{n}", "origen.json")
    if os.path.exists(ruta_origen) and json.load(open(ruta_origen, encoding="utf-8"))["huella"] == HUELLA:
        raise FileExistsError(f"data_estratificado{n} ya se hizo con estos mismos datos y parámetros. "
                              "No se crea otra carpeta.")

NOMBRE = f"data_estratificado{max(numeros, default=0) + 1}"
SALIDA = os.path.join(PROYECTO, NOMBRE)
TEMPORAL = os.path.join(PROYECTO, "_en_proceso_" + NOMBRE)
shutil.rmtree(TEMPORAL, ignore_errors=True)
print("Huella:", HUELLA[:16], "...")
print("Se creará:", SALIDA)

Huella: ec56b4baf1ccb02a ...
Se creará: /content/drive/MyDrive/tesis-rg6final/data_estratificado2


## 4) Leer y unir las anotaciones
Se unen train, valid y test, conservando solo las clases rg6Bien y rg6Mal.

In [15]:
imagenes = []
otras_clases, sin_clases = Counter(), 0

for carpeta in ["train", "valid", "test"]:
    with open(os.path.join(RUTA_ENTRADA, carpeta, "_annotations.coco.json"), encoding="utf-8") as f:
        coco = json.load(f)
    nombre_de = {c["id"]: c["name"] for c in coco["categories"]}

    anotaciones_por_imagen = {}
    for a in coco["annotations"]:
        clase = nombre_de[a["category_id"]]
        if clase in CLASES:
            anotaciones_por_imagen.setdefault(a["image_id"], []).append((clase, a))
        else:
            otras_clases[clase] += 1

    for im in coco["images"]:
        anots = anotaciones_por_imagen.get(im["id"], [])
        if not anots:
            sin_clases += 1
            continue
        n_bien = sum(1 for clase, _ in anots if clase == "rg6Bien")
        imagenes.append({"archivo": im["file_name"],
                         "base": im["file_name"].split(".rf.")[0],
                         "ruta": os.path.join(RUTA_ENTRADA, carpeta, im["file_name"]),
                         "ancho": im["width"], "alto": im["height"],
                         "anotaciones": anots, "n_bien": n_bien, "n_mal": len(anots) - n_bien})

print(f"Fotos con rg6Bien y/o rg6Mal: {len(imagenes)}")
print(f"Instancias -> rg6Bien: {sum(i['n_bien'] for i in imagenes)} | rg6Mal: {sum(i['n_mal'] for i in imagenes)}")
print(f"Fotos descartadas (sin estas clases): {sin_clases}")
print(f"Anotaciones de otras clases eliminadas: {dict(otras_clases) or 'ninguna'}")

Fotos con rg6Bien y/o rg6Mal: 1342
Instancias -> rg6Bien: 3135 | rg6Mal: 3011
Fotos descartadas (sin estas clases): 0
Anotaciones de otras clases eliminadas: ninguna


## 5) Revisar imágenes repetidas y casi idénticas
Se detectan imágenes duplicadas por MD5 y similares por hash perceptual.

In [16]:
def md5(ruta):
    with open(ruta, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def dhash(ruta):
    gris = np.asarray(Image.open(ruta).convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    bits = (gris[:, 1:] > gris[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def casi_iguales(a, b):
    hb = np.array(b, dtype=np.uint64)
    pares = []
    for i, h in enumerate(a):
        distintos = np.unpackbits((hb ^ np.uint64(h)).view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)
        pares += [(i, int(j), int(distintos[j])) for j in np.nonzero(distintos <= UMBRAL_CASI_IGUAL)[0]]
    return pares

for im in tqdm(imagenes, desc="MD5 + dHash"):
    im["md5"], im["dhash"] = md5(im["ruta"]), dhash(im["ruta"])

repetidas = {h: [im["archivo"] for im in imagenes if im["md5"] == h]
             for h, n in Counter(im["md5"] for im in imagenes).items() if n > 1}
huellas = [im["dhash"] for im in imagenes]
casi = [(imagenes[i]["archivo"], imagenes[j]["archivo"], d) for i, j, d in casi_iguales(huellas, huellas) if i < j]

print(f"Fotos repetidas (idénticas): {sum(len(v) for v in repetidas.values())}")
print(f"Pares casi idénticos (≤ {UMBRAL_CASI_IGUAL} bits): {len(casi)}")
for a, b, d in casi[:10]:
    print(f"   {d} bits  {a}  ~  {b}")
assert not repetidas, f"Hay fotos repetidas: quítalas en Roboflow y vuelve a exportar. Ejemplos: {list(repetidas.values())[:3]}"

MD5 + dHash:   0%|          | 0/1342 [00:00<?, ?it/s]

Fotos repetidas (idénticas): 0
Pares casi idénticos (≤ 5 bits): 2
   5 bits  antes_393_jpeg.rf.16a53716548507e95e74805da5c6f97e.jpg  ~  despues_917_jpeg.rf.50ea35eaedbf206ec23e13002aaa1912.jpg
   5 bits  agregado-78-_jpg.rf.8d4ccfea57cac9adda24fc1d68af2ac8.jpg  ~  agregado-77-_jpg.rf.6bfd29b8af9d68a3357a77cec85ba20b.jpg


## 6) Test fijo
Si se indica un test fijo, se reutiliza y sus imágenes se retiran de la entrada.

In [17]:
test_fijo = []
apartadas = []
if TEST_FIJO:
    carpeta_test = os.path.join(PROYECTO, TEST_FIJO, "test")
    with open(os.path.join(carpeta_test, "_annotations.coco.json"), encoding="utf-8") as f:
        coco_test = json.load(f)
    for im in tqdm(coco_test["images"], desc="Test fijo"):
        ruta = os.path.join(carpeta_test, im["file_name"])
        test_fijo.append({"archivo": im["file_name"], "base": im["file_name"].split(".rf.")[0],
                          "md5": md5(ruta), "dhash": dhash(ruta)})
    bases_test = {t["base"] for t in test_fijo}
    md5_test = {t["md5"] for t in test_fijo}
    en_test = lambda im: im["base"] in bases_test or im["md5"] in md5_test
    apartadas = [im["archivo"] for im in imagenes if en_test(im)]
    imagenes = [im for im in imagenes if not en_test(im)]
    print(f"Test fijo: {len(test_fijo)} fotos | apartadas de la entrada: {len(apartadas)} | quedan: {len(imagenes)}")
else:
    print("Sin test fijo: el test se elige en el reparto 70/20/10.")

Test fijo:   0%|          | 0/141 [00:00<?, ?it/s]

Test fijo: 141 fotos | apartadas de la entrada: 135 | quedan: 1207


## 7) Estrato de cada imagen
Se asigna a cada imagen un estrato según las clases que contiene.

In [18]:
for im in imagenes:
    if im["n_bien"] > 0 and im["n_mal"] > 0:
        im["estrato"] = "bien_y_mal"
    elif im["n_bien"] > 0:
        im["estrato"] = "solo_bien"
    else:
        im["estrato"] = "solo_mal"

print("Fotos por estrato:", dict(Counter(im["estrato"] for im in imagenes)))

Fotos por estrato: {'bien_y_mal': 839, 'solo_bien': 181, 'solo_mal': 187}


## 8) Reparto estratificado
Se reparten las imágenes en train, valid y test manteniendo la proporción de cada estrato.

In [19]:
estratos = [im["estrato"] for im in imagenes]
indices = list(range(len(imagenes)))

if TEST_FIJO:
    train_idx, valid_idx = train_test_split(indices, test_size=VALID / (TRAIN + VALID),
                                            random_state=SEMILLA, stratify=estratos)
    reparto = {"train": train_idx, "valid": valid_idx}
else:
    resto, test_idx = train_test_split(indices, test_size=TEST, random_state=SEMILLA, stratify=estratos)
    train_idx, valid_idx = train_test_split(resto, test_size=VALID / (TRAIN + VALID), random_state=SEMILLA,
                                            stratify=[estratos[i] for i in resto])
    reparto = {"train": train_idx, "valid": valid_idx, "test": test_idx}

for parte, idx in reparto.items():
    print(f"{parte:5s}: {len(idx):4d} fotos")
if TEST_FIJO:
    print(f"test : {len(test_fijo):4d} fotos (copia del test fijo)")

train:  938 fotos
valid:  269 fotos
test :  141 fotos (copia del test fijo)


## 9) Guardar el dataset
Se guarda cada partición en formato COCO con las clases 0 = rg6Bien y 1 = rg6Mal.

In [20]:
categorias = [{"id": i, "name": nombre, "supercategory": "rg6"} for i, nombre in enumerate(CLASES)]

for parte, idx in reparto.items():
    carpeta = os.path.join(TEMPORAL, parte)
    os.makedirs(carpeta)
    imgs_json, anots_json, id_anot = [], [], 1
    for id_img, i in enumerate(tqdm(sorted(idx), desc=parte), start=1):
        im = imagenes[i]
        shutil.copy2(im["ruta"], os.path.join(carpeta, im["archivo"]))
        imgs_json.append({"id": id_img, "file_name": im["archivo"], "width": im["ancho"], "height": im["alto"]})
        for clase, a in im["anotaciones"]:
            anots_json.append({"id": id_anot, "image_id": id_img, "category_id": CLASES.index(clase),
                               "bbox": a["bbox"], "area": a["area"], "segmentation": a["segmentation"],
                               "iscrowd": a.get("iscrowd", 0)})
            id_anot += 1
    with open(os.path.join(carpeta, "_annotations.coco.json"), "w", encoding="utf-8") as f:
        json.dump({"images": imgs_json, "annotations": anots_json, "categories": categorias}, f)
    print(f"{parte:5s}: {len(imgs_json)} fotos y {len(anots_json)} anotaciones guardadas")

if TEST_FIJO:
    shutil.copytree(carpeta_test, os.path.join(TEMPORAL, "test"))
    print(f"test : copiado de {TEST_FIJO}/test")

train:   0%|          | 0/938 [00:00<?, ?it/s]

train: 938 fotos y 4247 anotaciones guardadas


valid:   0%|          | 0/269 [00:00<?, ?it/s]

valid: 269 fotos y 1254 anotaciones guardadas
test : copiado de data_estratificado1/test


## 10) Verificación final
Se comprueba que no haya imágenes compartidas entre particiones y que las proporciones sean correctas.

In [21]:
huella_visual = {im["archivo"]: im["dhash"] for im in imagenes + test_fijo}
filas, contenido, archivos = [], {}, {}
for parte in ["train", "valid", "test"]:
    carpeta = os.path.join(TEMPORAL, parte)
    with open(os.path.join(carpeta, "_annotations.coco.json"), encoding="utf-8") as f:
        coco = json.load(f)
    assert [c["name"] for c in coco["categories"]] == CLASES
    faltan = [im["file_name"] for im in coco["images"] if not os.path.exists(os.path.join(carpeta, im["file_name"]))]
    assert not faltan, f"Faltan fotos en {parte}: {faltan[:3]}"
    contenido[parte] = {md5(os.path.join(carpeta, im["file_name"])) for im in coco["images"]}
    archivos[parte] = [im["file_name"] for im in coco["images"]]

    clases_de = {}
    for a in coco["annotations"]:
        clases_de.setdefault(a["image_id"], set()).add(a["category_id"])
    tipos = Counter("bien_y_mal" if len(c) == 2 else ("solo_bien" if 0 in c else "solo_mal") for c in clases_de.values())
    por_clase = Counter(CLASES[a["category_id"]] for a in coco["annotations"])
    n = len(coco["images"])
    filas.append({"parte": parte, "imágenes": n,
                  "% solo_bien": round(100 * tipos["solo_bien"] / n, 1), "% solo_mal": round(100 * tipos["solo_mal"] / n, 1),
                  "% bien_y_mal": round(100 * tipos["bien_y_mal"] / n, 1),
                  "inst. rg6Bien": por_clase["rg6Bien"], "inst. rg6Mal": por_clase["rg6Mal"],
                  "% rg6Bien": round(100 * por_clase["rg6Bien"] / len(coco["annotations"]), 1)})

tabla = pd.DataFrame(filas)
total = int(tabla["imágenes"].sum())
tabla.insert(2, "% del total", (100 * tabla["imágenes"] / total).round(1))
bien, mal = int(tabla["inst. rg6Bien"].sum()), int(tabla["inst. rg6Mal"].sum())
tabla.loc[len(tabla)] = {"parte": "total", "imágenes": total, "% del total": 100.0,
                         "inst. rg6Bien": bien, "inst. rg6Mal": mal, "% rg6Bien": round(100 * bien / (bien + mal), 1)}
print(tabla.to_string(index=False))
tabla.to_csv(os.path.join(TEMPORAL, "resumen_estratificacion.csv"), index=False)

fuga = (contenido["train"] & contenido["valid"]) | (contenido["train"] & contenido["test"]) | (contenido["valid"] & contenido["test"])
assert not fuga, f"Hay {len(fuga)} fotos idénticas en dos partes distintas"

casi_entre_partes = []
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    for i, j, d in casi_iguales([huella_visual[x] for x in archivos[a]], [huella_visual[x] for x in archivos[b]]):
        casi_entre_partes.append({"partes": f"{a}-{b}", "foto_1": archivos[a][i], "foto_2": archivos[b][j], "bits_distintos": d})
print(f"\nPares casi idénticos en partes distintas (revisar a mano): {len(casi_entre_partes)}")
for p in casi_entre_partes[:10]:
    print("  ", p)

with open(os.path.join(TEMPORAL, "duplicados.json"), "w", encoding="utf-8") as f:
    json.dump({"umbral_bits": UMBRAL_CASI_IGUAL, "repetidas_identicas": 0,
               "pares_casi_identicos_en_la_entrada": [{"foto_1": a, "foto_2": b, "bits_distintos": d} for a, b, d in casi],
               "pares_casi_identicos_entre_partes": casi_entre_partes}, f, indent=2, ensure_ascii=False)
print("\nSin fotos repetidas entre train/valid/test.")

parte  imágenes  % del total  % solo_bien  % solo_mal  % bien_y_mal  inst. rg6Bien  inst. rg6Mal  % rg6Bien
train       938         69.6         15.0        15.5          69.5           2175          2072       51.2
valid       269         20.0         14.9        15.6          69.5            617           637       49.2
 test       141         10.5         14.2        17.0          68.8            349           318       52.3
total      1348        100.0          NaN         NaN           NaN           3141          3027       50.9

Pares casi idénticos en partes distintas (revisar a mano): 1
   {'partes': 'valid-test', 'foto_1': 'agregado-77-_jpg.rf.6bfd29b8af9d68a3357a77cec85ba20b.jpg', 'foto_2': 'agregado-78-_jpg.rf.8d4ccfea57cac9adda24fc1d68af2ac8.jpg', 'bits_distintos': 5}

Sin fotos repetidas entre train/valid/test.


## 11) Registrar el origen
Se guarda un registro de cómo se creó la partición.

In [22]:
origen = {
    "carpeta": NOMBRE,
    "entrada": ENTRADA,
    "test_fijo": TEST_FIJO,
    "huella": HUELLA,
    "nota_huella": "SHA-256 de las anotaciones de entrada (y del test fijo) + parámetros del reparto. "
                   "Solo es un registro: identifica con qué datos exactos se hizo esta carpeta.",
    "clases": CLASES, "proporciones": [TRAIN, VALID, TEST], "semilla": SEMILLA,
    "imagenes": {f["parte"]: int(f["imágenes"]) for f in filas},
    "apartadas_por_test_fijo": len(apartadas),
    "pares_casi_identicos_entrada": len(casi),
    "pares_casi_identicos_entre_partes": len(casi_entre_partes),
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
}
with open(os.path.join(TEMPORAL, "origen.json"), "w", encoding="utf-8") as f:
    json.dump(origen, f, indent=2, ensure_ascii=False)

os.rename(TEMPORAL, SALIDA)

destino = os.path.join(INFORME, NOMBRE)
os.makedirs(destino, exist_ok=True)
for archivo in ["resumen_estratificacion.csv", "duplicados.json", "origen.json"]:
    shutil.copy2(os.path.join(SALIDA, archivo), destino)

print(f"Dataset listo: {SALIDA}")
print(f"   Resumen para el informe: {destino}")

Dataset listo: /content/drive/MyDrive/tesis-rg6final/data_estratificado2
   Resumen para el informe: /content/drive/MyDrive/tesis-rg6final/informe_para_codex/oe1/data_estratificado2


## 12) Siguiente paso
La partición queda lista para los cuadernos de entrenamiento.